1. Data Sizing && Extension (`mov`, `movzbl`, `movslq`)
   Cheatsheet Reference: ... C DATATYPES table
   - THE CONCEPT: The CPU's physical registers are 64 bits wide, but C programs
     use variables of different size (`char` is 1 byte, `int` is 4 bytes). The
     CPU needs specific instructions to safely move smaller data into larger 
     buckets without leaving garbage in the upper bits.
   - `movb/w/l/q`: Standard moves. The suffix enforces the size. `movl` is for
     32-bit `int`s, `movq` is for 64-bit pointers/longs.
   - `movzbl` (Zero-extend): Moves a 1-byte `char` into a 4-byte `int` space, 
     filling the top 24 bits with pure `0`s. Used for unsigned variables.
   - `movslq` (Sign-Extend): Moves a 32-bit `int` into a 64-bit register. It
     takes the sign bit (the MSB) of the 32-bit number and copies it across all
     the new upper bits. This ensures a negative number stay mathametically 
     negative in the larger 64-bit space. 


2. The Memory vs. Math Trap
   ... MEMORY ADDRESSING MODES formulas
   - The Concept: Parentheses `()` mjean "go to the RAM." But the CPU has an 
     Address Generation Unit that computes complex array logic incredibly fast:
     `Base + (Index * Scale) + Offset`.
   - `mov` (Memory access): If you use `movl (%rdi, %rsi, 4), %eax`, the CPU 
     does the math, walks out to the physical RAM, opens the locker at that
     address, and copies the data into `%eax`.
   - `lea` (Load Effective Address): The compiler's favourite cheatcode. 
     `leaq (%rdi, %rsi, 4), %rax` tells the CPU to do the exact same array math,
     but never touch the RAM. It simply intercepts the calculated address and
     drops it straight into `%rax`. It's used for lightning-fast pointer 
     arithmetic or multiplying by consants like 3, 5 or 9.

---

1. Math and Pointers
   These instrucitons actively change the data inside your registers.

   Remember the AT&T rule: SOURCE FIRST, DESTINATION LAST.
   - `addl` (Add long): Adds two 32-bit numbers.
      - Concept: `addl src, dst` means `dst = dst + src`. It automatically
        updates the hardware flags (like Carry or Overflow) if the math gets too
        big. 
      - ... `addl $5, %eax`     (Adds 5 to whatever's inside register %eax)  

   - `subl` (Subtract Long): Subtracts two 32-bit numbers.
      - Concept: `subl src, dst` means `dst = dst - src`. Notice the direction:
        it subtracts the first thing from the second thing.
      - ... `subl %ebx, %eax` (`%eax` -= `%ebx` and save it in `%eax`).

   - `leaq` (Load Effective Address - Quad): The compiler's favourite math hack.
      - Concept: It uses the memory-caclulation hardware 
        (`Base + Index*Scale + Offset`)   ... but it NEVER ACCESSES RAM. It just
        calculates the final number and drops it into a register.
      - ... `leaq 5(%rax, %rax, 2), %rbx` <=> `%rbx = %rax + (%rax * 2) + 5`...
        It did a multiply and an add in one clock cycle. 


2. The Flag Setters (Ghost Math)
   These instructions do math, but they THROW THE RESULT IN THE TRASH. Their 
   only job is to flip the invisible CPU wires (Flags) so you can make a 
   decision on the next line. 
   `cmpl` (Compare Long):
      - Concept: `cmpl a, b` calculates `b - a`. If result 0... flips Zero Flag
        (ZF). If the result is negative, it flips the SIGN FLAG (SF). It leaves
        the registers completely untouched.
      - Eaxmple: `cmpl $10, %eax`   ...     `%eax - $10`... If ... ZF becomes 1
        if `%eax = 10`... as the result will be `0`.

   `tesl` (Test Long):
      - Concept: `testl a, b` calculates the bitwise AND (`a & b`). It updates
        the Zeri abd Sign flags based on the result. 
        ... <-- don't understand... will check it out later. 


3. The Pathfinders (Jumps)
   These instructions read the flags that `cmp` or `test` just set. If the 
   condition is met, they change the Program Counter (PC) to teleport to a new
   line of code. 
   - `jmp` (Jump):
      - Concept: Unconditional. It doesn't check the flags. It just teleports.
        Used for `while(true)` or skipping over the `else` block after an `if`
        finishes.
   - `je` / `jne` (Jump Equal / Jump Not Equal)
      - Concept: They read the ZERO FLAG. `je` jumps if ZF is 1 (meaning the 
        last `cmp` resulted in 0, so the items were equal). `jne` jumps if ZF is
        0.
   - `jg` / `jge` (Jump Greater / Jump Greater-Equal)
      - Concept: Used for SIGNED integers. After `cmpl a, b`, `jg` will jump if
        `b > a`,
   - `jl` / `jle` (Jump Less / Jump Less-Equal)
      - Concept: Used for SIGNED integers. After `cmpl a, b`, `jl` will jump if
        `b < a`.
      ```
      cmpl %esi, %edi   # Calculates edi - esi (b - a)
      jl   .L4          # Jumps to .L4 if x < y
      ```          
   - `js` / `jns` (Jump Sign / Jump Not Sign):
      - Concept: Checks the SIGN FLAG directly. `js` jumps if the last 
        calculation was negative (MSB is 1). `jns` jumps if it was positive or
        zero (MSB is 0).


4. The Stack Contract (Function Calls)
   When a function (Caller) wants to start another function (Callee), the CPU 
   has to use the RAM (The Stack)

   - `pushq` (Push Quad):
      - Concept: Grows the stack downward. It subtracts 8 from the Stack Pointer
        (`%rsp`), then writes a 64-bit register's data into that new RAM slot. 
        Used to back up data before a function ruins it.
   - `popq` (Pop Quad):
      - Concept: Shrinks the stack upward. It reads the 8 bytes currently at the
        top of the stack into a register, then adds 8 to `%rsp` to "delete" that
        slot.
   - `call`:
      - Concept: The ultimate bookmark. It pushes the memory address of the
        very next line of code onto the stack, and then unconditionally `jmp`s 
        to the new function. 
   - `ret` (Return):
      - Concept: The trip home. It pops that bookmarked address off the top of
        the stack and unconditionally `jmp`s to it, perfectly resuming the
        original function exactly where it left off.           


---

   Because the CPU only has one set of physical hardware registers, sharing them
   is a massive problem. When Function A (the CALLER) is running and decides
   to pause and run Function B (the CALLEE), they both have to share the exact
   same silicon. If Function A leaves important data in a register and Function 
   B blindly overwrites it to do its own math, Function A's data is permanently
   destroyed. To prevent this chaos, system architects createda  strict 
   "contract" that divides the registers into two distinct categories.


Caller-Saved (The Scratchpads)
   These registers (like `%eax`, `%edi`, `%esi`, `%edx`) are free-for-all 
   scratchpads. The Callee is allowed to completely wipe these registers and use
   them for its own math the millisecond it starts running. Therefore, if the 
   Caller has important temporary data sitting in one of these registers, it is
   the Caller's responsibility to push that data onto the stack (save it) before
   making the `call` instruction. If the Caller doesn't back it up, they cannot
   complain when the data is gone after the function returns. 

Callee-Saved (The Safe Deposit Boxes)
   These registers (like `%rbx`, `%r12`, `%r13`) are guaranteed safe zones for
   long-lived values. The contract strictly dictates that when the Callee
   finishes running, these registers must hold the exact same values they had 
   when the function started. If the Calee wants to use these registers for its
   own math, it's the CALLEE's responsibility to push the Caller's original 
   values onto the stack at the very beginning of the function, and pop them 
   back right before hitting `ret`. The Caller can confidently put data in 
   these registers knowing it will survive the function call completely untouched. 




---

int addOne(int x) {
    return x + 1;
}

---

addOne:
    movl    %edi, %eax
    addl    $1  , %eax
    ret
                // copy `x` into the return register, then add 1. 

                                <-- ahh... so in this case... int array[] is a memory type!
int getPlusFirst(int array[], int i) {          
    return array[i] + array[0];
}

---

getPlusFirst:
    movslq  (%esi), %rsi
    movl    (%rdi,%rsi,4), %eax           // needs 64-bit index to calculate 64-bit address
    addl    (%rdi), %eax
    ret



    # FIX: Stretch index `i` to 64-bit so we can do RAM math safely,
    # FIX: Fetch array[i] using 64-bit base/index. Put 32-bit int in %eax
    # FIX: Fetch array[0] (dereference %rdi) and add it to %eax. 


    // BUG: Grabs the chopped-in-half pointer, not the data. 
    // Bug: Uses 32-bit registers for a 64-bit RAM address.    
    // Bug: Syntax error mixing `l` with 64-bit registers.

        // To get value at an address, you need parentheses... You want `movl    (%rdi), %eax`
        // must use %rdi because 32-bit memory address (%edi) doesn't exist...

        // CPU needs a 64-bit index to calculate a 64-bit memoery address... You
        // are trying to stretch the DATA you pulled out of the array into 64 
        // btis. But the C code only asks you to return a standard 32-bit `int`.
        // The Fix: You need to stretch the INDEX (`i`) before you do the memory
        // lookup, so the CPU's memory calculator doesn't crash. 
        // `movslq %esi, %rsi`


---
   ... The strict rule for `addl` (Add Long) is simply that your operands must 
   match the 32-bit "bucket size" (using `%e...` registers). You only use 
   parentheses to dereference when the C code explicitly forces you to interact
   with a pointer or an array out in main memory; otherwise, the compiler will
   keep everything fast and logical inside the registers. 

   ... In C, when you pass an array into a function (like `int array[]`), you
   aren't actually handing the CPU the entire array. C secretly converts that
   array into a POINTER to its very first element. 

   A pointer is literally just a 64-bit memory address... Therefore, `%rdi`...

   On the other hand, the variable `%esi = i` is passed by value. It isn't a 
   location; it's just a raw, 32-bit number (like 5 or 10)... That's why it sits
   comfortably in the 32-bit `%esi` register. 

---

int maxInt(int a, int b) {
    if (a >= b)
        return a;
    else
        return b;
}

---

maxInt:
    cmpl    %esi, %edi          # checks `a - b`. If `a >= b`, return `a`; ... 
    jge     .LA_ge_b
    movl    %esi, %eax     
    jmp     .L_done
.LA_ge_b:
    movl    %edi, %eax
    jmp     .L_done   
.L_done:
    ret


   ... local labels should always start with a period, like `.L1` or `.L_loop`.
   - Why? The dot tells the assembler that this is a LOCAL SYMBHOL... if forgot
     the dot, linker might mistake your lael for a globally accessible function
     ... like `main` or `maxInt`... The dot ensures the lkabel is invisible 
     outside of this specific function...

2. The "L" (Label):...
   
3. Human-Readable Names: ...
   - `.L_if_true:`
   - `.L_loop_start:`
   - `.L_return`   

int sumArray(int array[], int size) {
    int total = 0;
    for (int i = 0; i < size; i++) {
        total += array[i];
    }
    return total;
}

---

sumArray:
    movl    $0, %eax                    # int total = 0
    movl    $0, %edx                    # int i = 0
    jmp     .L_test
.L_loop
    addl    (%rdi, %rdx, 4), %eax       # total += array[i]
    addl    $1, %edx # i++
.L_test
    cmpl    %esi, %edx                  # if (i < size)
    jl      .L_loop
    ret



---
   ... `movl $0, %edx` automatically zero-extends the whole 64-bit `%rdx` 
   register, you could technically juse use `(%rdi, %rdx, 4)` and skip the 
   `movslq` entirely... 

---

void moveSmallerToFront(int array[], int loop) {
    int temp;
    if (array[loop] < array[0]) {
        temp = array[loop];
        array[loop] = array[0];
        array[0] = temp;
    }
}


---

    # int array[]   == %rdi
    # int loop      == %esi
    # array[loop] address   == %rcx
    # (%rcx) == movl ($rcx) %rdx == array[loop]
    
moveSmallerToFront:
    movslq  %esi, %rsi
    leaq    (%rdi,%rsi,4), %rcx      # address of `array[loop]`
    movl    (%rdi), %eax             # %eax == `array[0]`
    movl    (%rcx), %edx             # %edx == `array[loop]`
    cmpl    %eax, %edx               
    jge     .L_done
.L_if:
    movl    %eax, (%rcx)             
    movl    %edx, (%rdi)
.L_done
    ret


---

void incrementAll(int array[], int size) {
    int* ptr = array;
    int* end = array + size;

    while (ptr != end) {
        *ptr = *ptr + 1;
        ptr++;
    }
}



---
        # %rdi == int array[]
        # %esi == int size
        # %rdx == int* ptr      ||  movl %rdi, %rdx
        # %rcx == int* end      ||  movslq %esi, %rsi; movl (%rdi, %rsi, 4) %rcx
incrementAll:
    movq    %rdi, %rdx
    movslq  %esi, %rsi; 
    leaq    (%rdi, %rsi, 4), %rcx
    jmp     .L_test
.L_loop:
    addl    $1, (%rdx)      # *ptr = *ptr + 1          <-- (%rdx) <=> dereferencing
    addq    $4, %rdx        # ptr++                 
.L_test:
    cmpq    %rcx, %rdx
    jne     .L_loop
.L_done:
    ret



---
            # FIX: Copy the array pointer into ptr (%rdx)
            # FIX: ptr++ must use `addq` becaus pointer are quads. 

            # need to use `q` when handling all things memory related
            # check for missing commas everywhere during exams... 

---

   The simplest way to remember the difference is that `mov` FETCHES ACTUAL DATA
   FROM MEMORY, while `lea` JUST DOES THE MATH TO CALCULATE AN ADDRESS. If you
   write `movq 8(%rbx), %rax`, the CPU calculates the address `%rbx + 8`,
   physically travels out to RAM, grabs the value stoed there, and 


---

1. `test` (The Zero / Mask Checker)
   - ... `tesl %src, %dest`  performs a BITWISE AND (`dst & src`). It throws
     the result away but sets te ZF and SF.
   - C code:
        if (x == 0)     // tesl %edi, %edi      je      .L_if_zero


---


2. `call`, `pushq`, `popq` (The Stack Lifecycle)
   - How they work: `pushq src`: Subtracts 8 from the Stack Pointer (`%rsp`),
     then writes the 64-bit register into that new RAM slot. 
   - `call func`: Pushes the address of the next instruction onto the stack, 
     then jumps to `func`.

```c
int math(int x) {
    int y = getOtherNumber();   // call getOtherNumber
    return x + y;
}
```
   - When & Why to write them:
      - Why `pushq`/`popq`: Because `getOtherNumber()` might destroy the 
        registers. If `math()` was keeping `x` in `%rbx` (a Callee-Saved register)
        , ...
      - Why `call`: ... to invoke other functions. 


3. `imull`/`imulq` (Signed Multiplication)
   - How it works: `imull src, dst` calculates `dst = dst * src`
   ...

4. `sall`

---

```asm
testl   %esi, %esi
jle     .Ldone
```

```c
if (x == 0) goto done;
```

```asm
subq    $24, %rsp
movl    %edi, 12(%rsp)
call    increment
addq    $24, %rsp
ret
```

```c
// reserve stack frame
// store int argument x at stack offset 12
increment(...);
// free stack frame
```

// The function creates local stack storage, passes/uses something around
// `increment`, then restores `%rsp` before returning. 

```asm
pushq   %rbx
movq    %rdi, %rbx
call    randomise
movq    %rbx, %rdi
popq    %rbx
ret
```

---
`%rbx` is being used to preserve a value across a function call.

Explanation: `call randomise` may overwrite caller-saved registers, but `%rbx`
must be restored by callee-function before returning. So the function saves old
`%rbx`, uses `%rbx` as long-lived storage, then restores it. 

```asm
movq    %rsi, %rbp          # %rbp = %rsi
movq    8(%rsp), %rax       # %rax = 8+%rsp
imulq   %rax, %rbp          # %rbp = %rbp * %rax
```



# %rbp = %rsi
# %rax = 8+%rsp
# %rbp = %rsi * (8+%rsp)



```c
long math(long a, long b) {
    res
}


result = b;
a = *(rsp + 8);
result = result * a;
```

... add brackets to calculate the offset--`movq 8(%rdi), %rax`... the CPU is
   hardwired to treat those brackets as a dereference command... will calculate
   the address and immediately fetch the RAM contents... 


leaq    8(%rdi), %rax
// int* ptr = &arr[2];

```asm
movswl  %di, %edi
movl    %esi, %ecx
sall    %cl, %edi
movw    %di, 12(%rsp)       # <-- funn enough as it seems i've forgotten once again... this is for
                            #     storing' values uin the stack!
```


---
```c
x = (int)x << y;

```
   .. crams three completely different x86 quirks into just four lines of code. 
   ... tests your knowledge of REGISTER SLICES, TYPE CASTING and HARDCODED CPU
   RULE. 

   ... Assume the first argument `x` arrived in `%rdi` and the second argument
   `y` arrived in `%esi`.


1. The Stretch (`movswl`)
   - The .. The action: ... (Move with Sign-extend Word to Long). It takes the
     16-bit `short` from `%di`, stretches it into a 32-bit `int` in `%edi`, and
     copies the sign bit so negative numbers stay negative. 
   - C Equivalent: `int x_ext = (int)x` (where `x` was a `short`).

2. The Hardware Quirk (`movl` & `%cl`)
   - The Action: We want to shift `x` left by `y` places (`x << y`). But why did
     we move (`%esi`) into `%ecx` first? Why not just write `sall %esi, %edi`?
   - The Quirk: The physical silicon... hardcoded rule: IF YOU WANT TO SHIFT BY
     A VARIABLE AMOUNT, THAT VARIABLE MUST BE STORED IN THE `%cl` register. Yoy
     literally are not allowed to use any other register for the shift count. 
   - The Flow: We copy `y` into `%ecx`. Then we use `%cl` (which is just the
     bottom 8-bits of `%ecx`) to tell the `sall` instruction how many times to 
     shift `%edi` to the left. 

3. The Chop and Store (`movw`)
   - The Action: We did our math in a nice, safe 32-bit space (`%edi`). Now we
     are dine, and the C code want us to save the result as a 16-bit `short`
     local variable on the stack. 
   - THE CHOP: By using `movw` (Move Word) and reading from `%di`, the CPU just
     grabs the bottom 16 bits of our answer and completely ignores the top 16 
     bits.  
   - The sTORE: `12(%rsp)` means go to the Stack Pointer, move 12 bytes up, and
     drop the 16-bit answer into the RAM there...



```asm
movzwl  12(%rsp), %eax
ret
```
2 byte -> 4 byte
return (int)(unsigned short)local;

---

   ... how the CPU handles complex control flow, memory grids, and performance
   optimisation. ... 


1. Jump Tables (The `switch` statement Cheat Code)
   When you write a C `switch` statement with cases 0 through 5, the compiler
   could write a massive chain of `if/else` comparisons. But that is slow--if
   the user inputs `5`, the CPU has to check and fail cases 0, 1, 2, 3, and 4 
   first. 

   Instead, the compiler creates a JUMP TABLE.

   - The Concept: It creates a  secrete array in the `.rodata` (Read-only Data)
     section of the program. This array holds the actual memory addresses 
     (pointers) of where the code for Case 0, Case 1, Case 2, etc., lives. 
   - The Instruction: `jmp *.L4(, rdi, 8)`
      - `%rdi` holds your switch variable (e.g., the number `3`)
      - `8` is the scale because memory addresses (pointers) are 8 bytes long. 
      - `.L4` is the base address of the array. 
      - The `*` (Crucial): This means INDIRECT JUMP. It tells the CPU: "
        Calculate `.L4 + (%rdi * 8)`. Go to that address and return the 8-byte
        address inside it then teleport to that address..."


2. Stack Procedure (The Breadcrumb Trail)
   We covered this briefly before, but ... The CPU      
   - `call label`: Does two things simultaneouslu. It takes the address of the
     very next line of code (the return address) and pushes it onto the stack.
     Then it changes the PC to jump to the `label`.
   - `ret`: Does the exact opposie... pops the top 8 bytes off the stack, 
     assumes it is a memory address, and teleports the pC back to that exact spot...   


3. Array Elements (2D Arrays in 1D RAM)
   Physical RAM is a single, perfectly straight line of bytes... not a 2D grid.
   If you write `A[i][j]` in C, the CPU has to flatten grid into straight line
   using ROW-MAJOR ORDER...
   
   ... The formula...
      - K: The size of the data type (e.g., 4 bytes for an `int`). You multiply
        everything by $K$ to convert "items" into physical "bytes".


4. LOCALITY (The Cache Predictor)
   The CPU cache is incredibly small compared to RAM. To guess what data to keep
   in the cache and what to throw away, hardware engineers rely on two 
   fundamental human programming habits, known as Locality.
   - TEMPORAL LOCALITY (Time): If you access a variable now, you will probably
     access it again very soon.
      - Example: A loop counter `i` or a `sum` variable now, you will probably
        access it again very soon.
   - SPATIAL LOCALITY (Space): If you access a memory address now, you will
     probably access the memory addressess immediately next to it very soon.
      - ... Iterating through an array... arrays stored contiguously... CPU
        fetches a whole block of the array at once, assuming you'll ask for
        `arr[1]` next. ... 



---
2. Caller/Callee-Saved is just a "Gentleman's Agreement"
   Because the CPU doesn't enforce register saving, the humans writing the
   compilers had to come up with a treaty. This treaty is called the 
   `System V ABI` (Application Binary Interface).

   ... giant PDF document says: "... if write a compiler for OS... let's all 
   agree that `%rbx` should be protected by the Callee, and `%r10` should be
   protected by the Caller"...

   it is NOT verified by the hardware... only verified by the COMPILER. ...     


